# 📦 vLLM install 内訳の実測（CPU）

uv / wheel Dataset の効果を数字で決める。


In [ ]:
# vLLM install の内訳測定（GPU 不要）
# 目的: 起動短縮 A（uv）/ B（wheel Dataset）のどちらが効くかを実測で決める。
#   1. pip install vllm==0.32.0         … 現行の実測基準
#   2. pip download --no-deps         … 純粋な転送+書込（wheel 1 個）
#   3. uv pip install --system        … 解決+展開の短縮幅
#   4. pip download（依存込み）        … wheel Dataset 化した場合の総量
import importlib.metadata as md
import json
import os
import subprocess
import sys
import time

SPEC = "vllm==0.32.0"


def run(cmd, label, timeout=3600):
    t0 = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    dt = time.time() - t0
    print(f"[{label}] {dt:.1f}s  rc={r.returncode}")
    if r.returncode != 0:
        print(((r.stdout or "") + (r.stderr or ""))[-1500:])
    return dt, r


def du(path):
    total = 0
    n = 0
    for root, _, files in os.walk(path):
        for f in files:
            try:
                total += os.path.getsize(os.path.join(root, f))
                n += 1
            except OSError:
                pass
    return total, n


def vers():
    out = {}
    for p in ("torch", "torchaudio", "transformers", "vllm"):
        try:
            out[p] = md.version(p)
        except Exception:
            out[p] = None
    return out


res = {"spec": SPEC, "before": vers()}
print("before:", res["before"])
print("python:", sys.version.split()[0])

# 1. 現行と同じ pip install
res["pip_install_sec"], _ = run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-input", SPEC],
    "pip install")
res["after_pip"] = vers()

# 2. wheel 1 個の転送時間（依存解決を除く）
dt, _ = run([sys.executable, "-m", "pip", "download", "--no-cache-dir",
             "--no-deps", SPEC, "-d", "/tmp/w1"], "pip download no-deps")
res["pip_download_nodeps_sec"] = dt
res["pip_download_nodeps_bytes"], _ = du("/tmp/w1")

# 3. uv（vllm だけ外して、依存は既に入っている状態で測る）
run([sys.executable, "-m", "pip", "uninstall", "-y", "vllm"], "uninstall vllm")
run([sys.executable, "-m", "pip", "install", "-q", "--no-input", "uv"],
    "install uv")
dt, r = run(["uv", "pip", "install", "--system", "-q", SPEC], "uv install")
res["uv_install_sec"] = dt
res["uv_ok"] = r.returncode == 0
res["uv_out"] = ((r.stdout or "") + (r.stderr or ""))[-800:]
res["after_uv"] = vers()

# 4. 依存込みの総 wheel サイズ（Dataset 化の実現性）
dt, _ = run([sys.executable, "-m", "pip", "download", "--no-cache-dir",
             SPEC, "-d", "/tmp/wf"], "pip download all", timeout=5400)
res["pip_download_all_sec"] = dt
res["pip_download_all_bytes"], res["pip_download_all_files"] = du("/tmp/wf")
print("all wheels:", res["pip_download_all_bytes"] / 1e9, "GB",
      res["pip_download_all_files"], "files")
print(sorted(os.listdir("/tmp/wf"))[:20])

with open("/kaggle/working/uvbench.json", "w") as f:
    json.dump(res, f, indent=2, ensure_ascii=False)
print(json.dumps(res, indent=2, ensure_ascii=False)[:4000])
